In [1]:
import pandas as pd
import numpy as np
from grape import Graph
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Edge type prediction with NP-KG

TODO: find source of the edges (MR or PKL), save prediction probabilities

### Load KG

In [2]:
KG_PATH = '../../np-kg/resources/knowledge_graphs/'
EMBED_PATH = 'embeddings/'
DATA_PATH = '../data/'
DIR_PATH = 'results/'

In [3]:
npkg = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v3.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [4]:
npkg.get_number_of_connected_components()

(348, 1, 1089125)

In [5]:
npkg = npkg.remove_disconnected_nodes()

In [6]:
npkg.enable()

In [9]:
##get negative edges
n_edges = npkg.get_number_of_directed_edges()
n_neg_edges = int(n_edges*0.2)
n_neg_edges

1557384

In [25]:
negative_graph = npkg.sample_negative_graph(
    number_of_negative_samples=1000,
    use_scale_free_distribution=True,
    sample_edge_types=True,
    sample_edge_weights=False,
    random_state=42,
    source_nodes_prefixes=['chebi', 'napdi_srs_imports'],
    destination_nodes_prefixes=['chebi', 'napdi_srs_imports', 'pr']
)

In [21]:
npkg.get_edge_type_name_from_edge_type_id(0)

'ro:0002610'

In [26]:
negative_graph.get_number_of_directed_edges()

1000

In [27]:
negative_graph.get_number_of_nodes()

1089125

In [28]:
negative_graph.dump_edges(
    DATA_PATH+'negative_edges_1000.tsv',
    separator='\t',
    header = True,
    sources_column_number = 0,
    destinations_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

### Load embeddings

In [7]:
model_name = 'TransE_l1'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    transe_embedding_dict = pickle.load(filep)

In [8]:
model_name = 'ComplEx'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    complex_embedding_dict = pickle.load(filep)

In [9]:
model_name = 'DistMult'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    distmult_embedding_dict = pickle.load(filep)

### Load KG data as edges

In [10]:
filename = 'reference-data/reference_data_curie_edges_20240829.tsv'

In [11]:
refdf = pd.read_csv(DATA_PATH+filename, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1369 entries, 0 to 1368
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   source            1369 non-null   object
 1   target            1369 non-null   object
 2   interaction_type  1369 non-null   object
dtypes: object(3)
memory usage: 32.2+ KB


In [12]:
negative_edges = pd.read_csv(DATA_PATH+'negative_edges_1000.tsv', sep='\t')
negative_edges['relation'] = 'negative'
negative_edges = negative_edges[['subject', 'relation', 'object']]
negative_edges.head()

,subject,relation,object
0,chebi:137439,negative,chebi:36866
1,chebi:59997,negative,chebi:38193
2,chebi:193827,negative,pr:p01375
3,chebi:141716,negative,pr:p11509
4,chebi:92718,negative,chebi:26410


In [13]:
kg_edges = pd.read_csv(KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv', sep='\t')
kg_edges = kg_edges[['source', 'relation', 'target']]
kg_edges = kg_edges.rename(columns={'source': 'subject', 'target': 'object'})
kg_edges.head()

,subject,relation,object
0,mgi:1913508,rdfs:subClassOf,so:0001217
1,mgi:1913508,ro:0002160,ncbitaxon:10090
2,so:0001217,rdfs:subClassOf,so:0000704
3,so:0001217,ro:0000086,so:0000010
4,so:0001217,bfo:0000050,ncbitaxon:9606


In [14]:
kg_edges.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7786923 entries, 0 to 7786922
Data columns (total 3 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   subject   object
 1   relation  object
 2   object    object
dtypes: object(3)
memory usage: 178.2+ MB


In [15]:
##find edge types where subject is a napdi node and object is a chebi node, use uri_to_curie_map for mapping napdi uris to curies
napdi_file = '../data/chebi-extensions-constituents-NP-20240229.tsv'
napdi_df = pd.read_csv(napdi_file, sep='\t')
napdi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 681 entries, 0 to 680
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   constituent_name  681 non-null    object
 1   URI               681 non-null    object
dtypes: object(2)
memory usage: 10.8+ KB


In [16]:
uri_to_curie_file = '/home/sanya/np-kg/resources/knowledge_graphs/uri_to_curie_map.tsv'
##read to dictinary
uri_to_curie_map = {}
with open(uri_to_curie_file, 'r') as filep:
    for line in filep:
        uri, curie = line.strip().split('\t')
        uri_to_curie_map[uri] = curie

In [17]:
##add curies to napdi_df
napdi_df['curie'] = napdi_df['URI'].map(uri_to_curie_map)
napdi_df.head()

,constituent_name,URI,curie
0,12beta-acetoxycimigenol 3-o-beta-d-xylopyranoside,http://napdi.org/napdi_srs_imports:12beta_acet...,napdi_srs_imports:12beta_acetoxycimigenol_3_o_...
1,2'-o-acetylactein,http://napdi.org/napdi_srs_imports:2_o_acetyla...,napdi_srs_imports:2_o_acetylactein
2,2'-o-acetylcimicifugoside h1,http://napdi.org/napdi_srs_imports:2_o_acetylc...,napdi_srs_imports:2_o_acetylcimicifugoside_h1
3,23-epi-26-deoxyactein,http://purl.obolibrary.org/obo/CHEBI_70243,chebi:70243
4,23-o-acetylshengmanol,http://napdi.org/napdi_srs_imports:23_o_acetyl...,napdi_srs_imports:23_o_acetylshengmanol


In [18]:
##find kg_edges where subject is a napdi node and object is a chebi node
kg_edges_napdi = kg_edges.merge(napdi_df, left_on='subject', right_on='curie', how='inner')
kg_edges_napdi = kg_edges_napdi[['subject', 'relation', 'object']]
kg_edges_napdi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 36868 entries, 0 to 36867
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   36868 non-null  object
 1   relation  36868 non-null  object
 2   object    36868 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB


In [19]:
kg_edges_napdi = kg_edges_napdi.loc[kg_edges_napdi['object'].str.contains('chebi')]
kg_edges_napdi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7374 entries, 5 to 36865
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   7374 non-null   object
 1   relation  7374 non-null   object
 2   object    7374 non-null   object
dtypes: object(3)
memory usage: 230.4+ KB


In [20]:
rel_types = kg_edges_napdi.relation.value_counts()
rel_types[:50]

rdfs:subClassOf                                1515
ro:0002180                                     1110
ro:0000087                                      791
ro:0002449                                      780
ro:0002434                                      710
ro:0002490                                      570
ro:0002448                                      331
ro:0002213                                      268
ro:0002448|ro:0002449                            96
ro:0002434|ro:0002449                            96
chebi#has:functional:parent                      92
chebi#is:conjugate:acid:of                       77
ro:0002434|ro:0002490                            73
ro:0011010                                       41
ro:0002213|ro:0002449                            40
chebi#has:parent:hydride                         39
ro:0011009                                       36
ro:0002434|ro:0002449|ro:0002490                 36
ro:0002213|ro:0002434                            34
ro:0002434|r

In [21]:
#rel_subset = [
#    'ro:0002449', 'ro:0002434', 'ro:0002448', 'ro:0002213', 'ro:0002448|ro:0002449', 'ro:0002434|ro:0002449', 'ro:0002434|ro:0002490', 'ro:0002213|ro:0002449',
#    'ro:0002434|ro:0002449|ro:0002490', 'ro:0002213|ro:0002434','ro:0002434|ro:0002448', 'ro:0002213|ro:0002434|ro:0002449','ro:0002449|ro:0002490',
#    'ro:0002213|ro:0002448','ro:0002596','ro:0002213|ro:0002434|ro:0002449|ro:0002490']
rel_subset = ['ro:0002449', 'ro:0002434', 'ro:0002448', 'ro:0002213']

In [22]:
kg_edges_napdi_subset = kg_edges_napdi.loc[kg_edges_napdi['relation'].isin(rel_subset)]
kg_edges_napdi_subset = kg_edges_napdi_subset.reset_index(drop=True)
kg_edges_napdi_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2089 entries, 0 to 2088
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   2089 non-null   object
 1   relation  2089 non-null   object
 2   object    2089 non-null   object
dtypes: object(3)
memory usage: 49.1+ KB


In [23]:
kg_edges_combined = pd.concat([kg_edges_napdi_subset, negative_edges], ignore_index=True)
kg_edges_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3089 entries, 0 to 3088
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   3089 non-null   object
 1   relation  3089 non-null   object
 2   object    3089 non-null   object
dtypes: object(3)
memory usage: 72.5+ KB


In [24]:
kg_edges_combined.head()

,subject,relation,object
0,chebi:16796,ro:0002449,chebi:131715
1,chebi:16796,ro:0002434,chebi:35204
2,chebi:16118,ro:0002449,chebi:175111
3,chebi:16118,ro:0002434,chebi:55323
4,chebi:16118,ro:0002213,chebi:35720


In [25]:
##get counts
kg_edges_combined['relation'].value_counts()

negative      1000
ro:0002449     780
ro:0002434     710
ro:0002448     331
ro:0002213     268
Name: relation, dtype: int64

In [26]:
kg_edges_combined.drop_duplicates(inplace=True)
kg_edges_combined.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2795 entries, 0 to 3088
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   2795 non-null   object
 1   relation  2795 non-null   object
 2   object    2795 non-null   object
dtypes: object(3)
memory usage: 87.3+ KB


In [27]:
le = LabelEncoder()
kg_edges_combined['relation_encoded'] = le.fit_transform(kg_edges_combined['relation'])

In [28]:
kg_edges_combined.subject.nunique()

1108

In [29]:
kg_edges_combined.object.nunique()

1510

In [30]:
kg_edges_combined.relation.value_counts()

negative      1000
ro:0002449     670
ro:0002434     588
ro:0002448     310
ro:0002213     227
Name: relation, dtype: int64

In [34]:
kg_edges_combined.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2795 entries, 0 to 3088
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   subject           2795 non-null   object
 1   relation          2795 non-null   object
 2   object            2795 non-null   object
 3   relation_encoded  2795 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 109.2+ KB


### Edge Type Prediction

In [31]:
def get_embeddings_df(embedding_dict):
    node_embeddings_df = embedding_dict['node_embeddings'][0]
    print(f"Node Embeddings Shape: {node_embeddings_df.shape}")
    return node_embeddings_df

In [37]:
def extract_features(edges, node_embeddings_df, edge_feature_method='concat'):
    features = []
    for subj, obj, pred, pred_encoded in edges:
        try:
            node_subj_embedding = node_embeddings_df.loc[subj].values
            node_obj_embedding = node_embeddings_df.loc[obj].values
            if edge_feature_method == 'concat':
                node_feature = np.concatenate([node_subj_embedding, node_obj_embedding])
            elif edge_feature_method == 'hadamard':
                node_feature = np.multiply(node_subj_embedding,node_obj_embedding)  # Hadamard product
            features.append(node_feature)
        except KeyError:
            print(f'KeyError: {subj}, {obj}, {pred}')
            continue
    print(f"Features Shape: {np.array(features).shape})")
    return np.array(features)

In [33]:
model_name = 'ComplEx'
if model_name == 'TransE':
    embedding_dict = transe_embedding_dict
elif model_name == 'ComplEx':
    embedding_dict = complex_embedding_dict
elif model_name == 'DistMult':
    embedding_dict = distmult_embedding_dict

In [36]:
node_embeddings_df = get_embeddings_df(embedding_dict)
edge_feature_method = 'concat'
# Prepare the data
kg_edges_array = kg_edges_combined[['subject', 'object', 'relation', 'relation_encoded']].values
kg_edges_array.shape

Node Embeddings Shape: (1089125, 800)


(2795, 4)

In [38]:
X_kg = extract_features(kg_edges_array, node_embeddings_df, edge_feature_method)
y_kg = kg_edges_array[:, 3].astype(int)
print(f"X_kg Shape: {X_kg.shape}, y_kg Shape: {y_kg.shape}")
print(f"Unique Labels in y_kg: {np.unique(y_kg)}")
edge_type_classifier = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Features Shape: (2795, 1600))
X_kg Shape: (2795, 1600), y_kg Shape: (2795,)
Unique Labels in y_kg: [0 1 2 3 4]


In [39]:
edge_type_accuracies = []
edge_type_precisions = []
edge_type_recalls = []
edge_type_f1s = []
edge_type_aucs = []
edge_type_avg_precision = []
predictions = []
pred_probs = []
for train_index, test_index in kf.split(X_kg, y_kg):
    X_train, X_test = X_kg[train_index], X_kg[test_index]
    y_train, y_test = y_kg[train_index], y_kg[test_index]
    edge_type_classifier.fit(X_train, y_train)
    y_pred = edge_type_classifier.predict(X_test)
    y_proba = edge_type_classifier.predict_proba(X_test)
    ##save predictions and probabilities
    #predictions.append(y_pred)
    #pred_probs.append(y_proba)
    edge_type_accuracies.append(accuracy_score(y_test, y_pred))
    edge_type_precisions.append(precision_score(y_test, y_pred, average='weighted'))
    edge_type_recalls.append(recall_score(y_test, y_pred, average='weighted'))
    edge_type_f1s.append(f1_score(y_test, y_pred, average='weighted'))
    edge_type_aucs.append(roc_auc_score(y_test, y_proba, multi_class='ovr'))
    edge_type_avg_precision.append(average_precision_score(y_test, y_proba, average='weighted'))

print("Edge-Type Prediction Accuracy: {:.2f} ± {:.2f}".format(np.mean(edge_type_accuracies), np.std(edge_type_accuracies)))
print("Edge-Type Prediction Precision: {:.2f} ± {:.2f}".format(np.mean(edge_type_precisions), np.std(edge_type_precisions)))
print("Edge-Type Prediction Recall: {:.2f} ± {:.2f}".format(np.mean(edge_type_recalls), np.std(edge_type_recalls)))
print("Edge-Type Prediction F1 Score: {:.2f} ± {:.2f}".format(np.mean(edge_type_f1s), np.std(edge_type_f1s)))
print("Edge-Type Prediction AUC: {:.2f} ± {:.2f}".format(np.mean(edge_type_aucs), np.std(edge_type_aucs)))
print("Edge-Type Prediction Average Precision: {:.2f} ± {:.2f}".format(np.mean(edge_type_avg_precision), np.std(edge_type_avg_precision)))

Edge-Type Prediction Accuracy: 0.67 ± 0.01
Edge-Type Prediction Precision: 0.68 ± 0.01
Edge-Type Prediction Recall: 0.67 ± 0.01
Edge-Type Prediction F1 Score: 0.67 ± 0.01
Edge-Type Prediction AUC: 0.87 ± 0.01
Edge-Type Prediction Average Precision: 0.69 ± 0.01


In [ ]:
###Fit model on all data in kg_edges_combined and save model to run on reference data
edge_type_classifier.fit(X_kg, y_kg)

### Predict edge types in reference data

In [ ]:
# Prepare the reference data
ref_edges = refdf[['source', 'target']].values

##original reference data with interaction type -- inhibition/induction/unknown